## Exercise — Loan-Approval Fairness Audit

Run a focused fairness audit on the loan-approval model. The scaffold below expects you to compute the three core metrics **from confusion-matrix primitives first** (~15 lines, matching the capstone's `lab/fairness_metrics.py` shape), then optionally cross-check with Fairlearn.

Section 7 is an **optional GenAI bonus** — once the tabular audit is done, drop the same primitives onto a small synthetic LLM-hiring-screen fixture (`data/llm_hiring_screen.csv`).

**Color guide (notebook callouts):**
- Markdown cells starting with `📌 TODO` are your work areas.
- Markdown cells starting with `🔒 Reference` are read-only context.
- Code cells with `# TODO` need filling in.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fairlearn.metrics import (
    MetricFrame, demographic_parity_ratio, equalized_odds_difference, true_positive_rate
)
from sklearn.metrics import accuracy_score

np.random.seed(42)
df = pd.read_csv("data/loan_predictions.csv")
df.head()

## 1. Compute the three core metrics across each subgroup column

In [ ]:
# 📌 TODO Step 1: Implement the three primitives below (~15 lines total).
# Same shape as the capstone's lab/fairness_metrics.py — write them once,
# reuse them on the project.

def selection_rate(y_pred):
    # TODO: fraction of predictions that are positive (one line)
    pass

def tpr(y_true, y_pred):
    # TODO: positive class only — what fraction of true positives did we predict positive?
    pass

def fpr(y_true, y_pred):
    # TODO: negative class only — what fraction of true negatives did we wrongly flip to 1?
    pass

def audit_one_subgroup(df, subgroup_col, score_col='prediction', label_col='label'):
    # TODO: build per-group rates, return
    #   {dp_ratio, eo_diff, eopp_diff, accuracy}
    # dp_ratio  = min(group selection rates) / max(group selection rates)
    # eo_diff   = max gap across groups in EITHER TPR or FPR
    # eopp_diff = max gap across groups in TPR only
    pass

# This builds the `results` dict downstream cells rely on. Once you implement
# `audit_one_subgroup` above, this line will produce real metrics; until then,
# it will raise a clear error pointing back to your unfinished TODO.
results = {col: audit_one_subgroup(df, col) for col in ['gender', 'race', 'age_band']}
assert all(m is not None for m in results.values()), (
    "audit_one_subgroup() returned None — implement the TODO in the function above "
    "before running the downstream cells."
)

# Optional one-line Fairlearn cross-check on age_band:
# from fairlearn.metrics import demographic_parity_ratio
# fl_dp_age = demographic_parity_ratio(y_true=df.label, y_pred=df.prediction,
#                                      sensitive_features=df.age_band)


## 2. Status table against policy thresholds


In [ ]:
POLICY = {"dp_ratio": 0.80, "eo_diff": 0.10, "eopp_diff": 0.10}

def status_table(results):
    rows = []
    for col, m in results.items():
        rows.append({
            "subgroup": col,
            "DP ratio":  m["dp_ratio"],
            "DP pass":   m["dp_ratio"] >= POLICY["dp_ratio"],
            "EO diff":   m["eo_diff"],
            "EO pass":   m["eo_diff"] <= POLICY["eo_diff"],
            "EOpp diff": m["eopp_diff"],
            "EOpp pass": m["eopp_diff"] <= POLICY["eopp_diff"],
            "Accuracy":  m["accuracy"],
        })
    return pd.DataFrame(rows)

status_table(results)

## 3. Intervention — per-group threshold adjustment

The model produces a continuous `score` (in `df.score`). The current 0/1 prediction uses a global threshold of 0.50. We can ship per-group thresholds that pull the failing metric back inside the policy bound — at some accuracy cost.

In [ ]:
def per_group_thresholds(df, subgroup_col, target_pos_rate):
    """Pick a per-group threshold so each subgroup's predicted-positive rate ≈ target_pos_rate.

    Args:
        df:               DataFrame with 'score' and subgroup_col.
        subgroup_col:     subgroup column to balance positive rates across.
        target_pos_rate:  shared target positive rate across groups (use overall positive rate).

    Returns:
        dict {group: threshold}. Each group's threshold is the (1 - target_pos_rate) quantile of its scores.
    """
    # TODO Step 1: for each group g, threshold = group's score quantile at (1 - target_pos_rate).
    # TODO Step 2: return {group: threshold} dict.
    pass


def apply_per_group_thresholds(df, subgroup_col, thresholds):
    """Return a new prediction column from per-group thresholds."""
    out = df["prediction"].copy()
    for g, thr in thresholds.items():
        mask = df[subgroup_col] == g
        out.loc[mask] = (df.loc[mask, "score"] >= thr).astype(int)
    return out

# Apply intervention on the worst-performing subgroup column.
# (Pick the column with the lowest dp_ratio.)
worst_col = min(results, key=lambda c: results[c]["dp_ratio"])
print(f"Worst-performing subgroup column: {worst_col}")

target = float(df.prediction.mean())
thr = per_group_thresholds(df, worst_col, target)
print(f"Per-group thresholds: {thr}")

df_after = df.copy()
df_after["prediction"] = apply_per_group_thresholds(df, worst_col, thr)

# Re-audit the same subgroup column after the intervention.
after_metrics = audit_one_subgroup(df_after, worst_col)
print("AFTER:", after_metrics)

## 4. Quantify the trade-off

In [ ]:
before = results[worst_col]
after  = after_metrics

print(f"Subgroup column audited: {worst_col}")
print(f"  DP ratio:  {before['dp_ratio']:.3f} → {after['dp_ratio']:.3f}  (bound ≥ 0.80)")
print(f"  EO diff:   {before['eo_diff']:.3f} → {after['eo_diff']:.3f}    (bound ≤ 0.10)")
print(f"  EOpp diff: {before['eopp_diff']:.3f} → {after['eopp_diff']:.3f}  (bound ≤ 0.10)")
print(f"  Accuracy:  {before['accuracy']:.3f} → {after['accuracy']:.3f}")

## 5. Render the metrics-comparison chart

In [ ]:
# TODO Step 6: Render a 4-panel grouped bar chart comparing the BEFORE and AFTER
# fairness metrics for the worst-performing subgroup column. Specification:
#
# - Panel layout: 4 side-by-side panels, one per metric. Order them:
#     Panel 1 — DP ratio        (policy bound: ≥ 0.80)
#     Panel 2 — EO diff         (policy bound: ≤ 0.10)
#     Panel 3 — EOpp diff       (policy bound: ≤ 0.10)
#     Panel 4 — Accuracy        (no policy bound — just compare the two bars)
# - Each panel shows TWO bars: BEFORE and AFTER. Use the `before` and `after`
#   dicts already computed in section 4.
# - Draw a red dashed horizontal line on each of panels 1-3 at the policy bound.
#   Use different orientation if helpful (DP is a floor, EO/EOpp are ceilings).
# - Title: "UdaciBank Loan Model — Fairness Audit (BEFORE vs AFTER per-group threshold adjustment)"
# - Save to "fairness_metrics_chart.png" at 150 dpi.
#
# Hint: `matplotlib.pyplot` is already imported as `plt`. The solution uses a
# 1-row-by-4-column figure with the same color palette across panels (one color
# for BEFORE, another for AFTER) and a shared legend.


## 6. Launch Recommendation Memo (fill in below)

**Recommendation:** [Launch / Conditional Launch / No-Go]

**Rationale:** [1 paragraph — tie back to the BEFORE/AFTER metrics above and the policy thresholds.]

**Recommended controls:**
1. [Your response here — name the per-group thresholds you set, or the alternative.]
2. [Your response here — monitoring control.]
3. [Your response here — retraining cadence / governance gate.]

**90-day monitoring plan:**
- Metrics monitored: [list]
- Cadence: [weekly / monthly]
- Owner: [named role]
- Escalation: [paging path on policy breach]

---

**Key takeaway (do not edit):** fairness metrics are inputs to a governance decision, not the decision itself. A fairness audit is incomplete without (a) intervention testing and (b) a launch recommendation. The deliverable is a defensible decision, not a metric table.

## 7. (Optional Bonus) Section 7 — Same primitives, GenAI surface

📌 **TODO Step 7 (optional):** Once the tabular audit is done, point the same `audit_one_subgroup()` at the synthetic LLM-hiring-screen fixture in `data/llm_hiring_screen.csv`. The audit math is identical — what changes is the input pipeline.

🔒 **Reference:** The tabular spine is the regulatory-grounded primary lesson (ECOA / Reg B, NYC LL-144, EU AI Act Annex III). Section 7 simply shows the primitives transfer to a GenAI surface without losing the policy interpretation anchor.

**Further reading for the GenAI side:** BBQ, BiasInBios, HELM (LLM-fairness benchmarks). For cross-method validation: AIF360, Aequitas.

In [ ]:
# TODO Step 7 (optional): Load llm_hiring_screen.csv, rename columns, and run
# audit_one_subgroup() with subgroup_col='gender'. Print the policy-check verdict.

# llm = pd.read_csv('data/llm_hiring_screen.csv')
# llm_audit = audit_one_subgroup(
#     llm.rename(columns={'llm_recommendation': 'prediction',
#                         'human_label': 'label'}),
#     subgroup_col='gender'
# )
# print(llm_audit)


## 8. (Optional Stretch) Part 3 Bonuses

The README lists two optional Part 3 stretches that build on the audit-and-intervention pipeline above. Pick whichever is more relevant to your team's actual risk posture.

📌 **Stretch A — Add predictive parity as a fourth metric.** Predictive parity (a.k.a. equal predictive value) compares per-group positive predictive value: among the people the model flagged positive, what fraction were actually positive across each group? The math sits in the same shape as your `audit_one_subgroup` primitive — add a `ppv_diff` field. Decide what policy threshold you'd set for it and explain your choice in 1-2 sentences in the memo.

📌 **Stretch B — Test `ThresholdOptimizer` as a second intervention.** Fairlearn's `ThresholdOptimizer` is post-processing per-group threshold optimization with a specified fairness constraint (e.g., `constraints="equalized_odds"`). Run it on the same `worst_col` you used for section 3, and report whether it reaches the same / better / worse fairness-vs-accuracy trade-off than your simple per-group quantile thresholding. The Reference Notes cell below covers the post-processing-vs-pre-processing distinction worth mentioning in your memo.

Treat both as memo-grade extensions — produce a short numeric comparison + a one-paragraph interpretation, not a new chart.


## Reference Notes

A few specification details for the fairness-policy thresholds and library APIs used above:

- **0.10 ceiling for equalized-odds and equal-opportunity differences.** The 0.10 threshold used in the SafeWheels / UdaciBank scenarios is a **firm-policy threshold** chosen for this lesson, not a regulatory or industry-wide standard. Real organizations set EO / EOpp ceilings based on their policy, regulatory exposure, and the distribution of their training data; common firm policies range from 0.05 to 0.15 depending on use case and risk tier. By contrast, the **0.80 demographic-parity floor** is anchored to the EEOC 4/5ths rule and has external regulatory provenance.
- **`ThresholdOptimizer` is post-processing per-group threshold optimization.** It selects a separate decision threshold per protected group to satisfy a fairness constraint (e.g., equalized odds). It does not modify the training data — that's the role of pre-processing techniques like AIF360's `Reweighing`. Both families are valid mitigations; they intervene at different points in the pipeline.
- **The fairness-impossibility result.** The formal impossibility result (Chouldechova 2017; Kleinberg–Mullainathan–Raghavan 2016) is between **calibration** and **error-rate balance under unequal base rates**. The "impossibility trilemma" framing across DP, EO, and EOpp is a related practical observation — these metrics generally cannot all hold simultaneously except in degenerate cases — but the formal proof targets the calibration-vs-equalized-odds pair specifically.